In [2]:
# To be able to make edits to repo without having to restart notebook
%load_ext autoreload
%autoreload 2

In [3]:
import os
from AxonaDataReader import AxonaDataReader, extract_MECO1_cut_file_meta_data
import xarray as xr

/Users/Anurag/Downloads/NeuroLab/NSK-isolated


In [2]:
def save_xarrays(session_xarrays, save_dir):

    os.makedirs(save_dir, exist_ok=True)

    for i, session in enumerate(session_xarrays):
        # session_path = os.path.join(save_dir, f"session_{i+1}")
        # os.makedirs(session_path, exist_ok=True)  # Create a subdirectory for each session

        for key, xarr in session.items():
            if isinstance(xarr, xr.Dataset) or isinstance(xarr, xr.DataArray):  # If it's a single Xarray, save it
                data_name = xarr.attrs.get("data_name")
                save_path = os.path.join(save_dir, f"{data_name}_{key}.nc")
                if key == "pos_array" and isinstance(xarr, xr.Dataset):
                    xarr = xarr.to_dataarray(name="animal_position")
                xarr.to_netcdf(save_path)
                print(f"Saved {key} to {save_path}")
            elif isinstance(xarr, list) and key!="meta_data":  # If it's a list of Xarrays, save each separately
                for j, xarr_item in enumerate(xarr):
                    data_name = xarr_item.attrs.get("data_name")
                    save_path = os.path.join(save_dir, f"{data_name}_{key}.nc")
                    xarr_item.to_netcdf(save_path)
                    print(f"Saved {key}_{j+1} to {save_path}")



In [8]:
import numpy as np

data_directory = "/Users/Anurag/Downloads/NeuroLab/Data/GusRemapData"  # Change this to your actual data folder
    
# Check if the directory exists
if not os.path.isdir(data_directory):
    raise FileNotFoundError(f"Data directory '{data_directory}' does not exist.")

subdirs = np.sort([ f.path for f in os.scandir(data_directory) if f.is_dir() ])
    # Initialize AxonaDataReader with a metadata extractor function
reader = AxonaDataReader(meta_data_extractor=extract_MECO1_cut_file_meta_data)

session_xarray_list = []
for subdir in subdirs:
    # Read all sessions and get Xarray objects
    print(f'converting to netcdf for mice: {subdir.split('/')[-1]}')
    session_xarrays = reader._read_batch_session(subdir)
    session_xarray_list.append(session_xarrays)

    # Print the structure of the first session for verification
    if session_xarrays:
        print("Successfully read sessions! Structure of the first session:")
        for key, xarr in session_xarrays[0].items():
            print(f"{key}: {type(xarr)}")
            if isinstance(xarr, list):
                print(f"  Contains {len(xarr)} elements")

converting to netcdf for mice: 1_13
DECODING PPM FROM FILE
PPM HERE:  589.0
DECODING PPM FROM FILE
PPM HERE:  589.0
DECODING PPM FROM FILE
PPM HERE:  589.0
DECODING PPM FROM FILE
PPM HERE:  589.0
Successfully read sessions! Structure of the first session:
pos_array: <class 'xarray.core.dataarray.DataArray'>
set_array: <class 'NoneType'>
cut_arrays: <class 'list'>
  Contains 4 elements
tet_waveform_arrays: <class 'list'>
  Contains 4 elements
event_time_arrays: <class 'list'>
  Contains 4 elements
meta_data: <class 'list'>
  Contains 4 elements
converting to netcdf for mice: 1_14
DECODING PPM FROM FILE
PPM HERE:  589.0
DECODING PPM FROM FILE
PPM HERE:  589.0
DECODING PPM FROM FILE
PPM HERE:  589.0
DECODING PPM FROM FILE
PPM HERE:  589.0
Successfully read sessions! Structure of the first session:
pos_array: <class 'xarray.core.dataarray.DataArray'>
set_array: <class 'NoneType'>
cut_arrays: <class 'list'>
  Contains 3 elements
tet_waveform_arrays: <class 'list'>
  Contains 3 elements
even

In [12]:
print(session_xarrays[0]["tet_waveform_arrays"][0].attrs.get("data_name"))
print(type(session_xarrays[0]['pos_array']))

1-13_1_20210510_163938
<class 'xarray.core.dataarray.DataArray'>


In [6]:
session_xarrays[0]['tet_waveform_arrays'][0]

<xarray.DataArray (spike_idx: 136831, channel: 4, sample: 50)> Size: 109MB
array([[[ 18.,  34.,  43., ...,  20.,  16.,   6.],
        [ 10.,  31.,  58., ...,  30.,  26.,  16.],
        [ 18.,  34.,  45., ...,   0.,  -5., -15.],
        [ 28.,  40.,  46., ...,   9.,   0., -11.]],

       [[ 18.,  14.,  13., ..., -41., -53., -61.],
        [ 15.,   9.,   4., ..., -13., -16., -26.],
        [ 19.,  25.,  37., ..., -24., -33., -39.],
        [ 28.,  26.,  24., ..., -23., -36., -43.]],

       [[ 21.,  10.,   4., ..., -19., -12.,  -2.],
        [ 27.,   7.,  -7., ..., -34., -14.,   3.],
        [ 19.,   8.,  -1., ..., -11.,   7.,  24.],
        [  7.,  -5.,  -6., ..., -13.,  -2.,   9.]],

       ...,

       [[ 56.,  56.,  45., ...,  21.,  26.,  25.],
        [ 51.,  55.,  52., ...,  24.,  28.,  31.],
        [ 45.,  33.,  21., ...,   9.,  12.,  20.],
        [ 50.,  53.,  48., ...,   0.,   5.,  12.]],

       [[ 22.,  22., -33., ..., -32., -24.,  -9.],
        [ 35.,  13., -17., ..., -38., -35., -21.],
        [ 14., -20., -51., ..., -13.,  -2.,  10.],
        [  1., -23., -41., ..., -36.,  -3.,  37.]],

       [[-29., -19.,  -9., ..., -10.,   1.,   6.],
        [-31., -28., -15., ...,   3.,   7.,  12.],
        [-40., -37., -30., ...,  14.,  18.,  10.],
        [-32., -19.,  -5., ...,  -6.,  -2.,  -8.]]], dtype=float32)
Coordinates:
  * spike_idx  (spike_idx) float32 547kB 0.0 1.0 2.0 ... 1.368e+05 1.368e+05
  * channel    (channel) float32 16B 0.0 1.0 2.0 3.0
  * sample     (sample) float32 200B 0.0 1.0 2.0 3.0 4.0 ... 46.0 47.0 48.0 49.0
Attributes:
    session_data_ref:      {'schema_ref': 'session', 'data_name': '1a40_20230...
    animal_data_ref:       {'schema_ref': 'animal', 'data_name': '1a40'}
    probe_data_ref:        {'schema_ref': 'probe', 'data_name': '1'}
    has_file:              true
    schema_ref:            tet_waveforms
    data_name:             1a40_1_20230119_151319
    data_dimensions:       ['spike_idx', 'channel', 'sample']
    dimension_of_measure:  [charge]

In [21]:
session_xarrays[0]['pos_array']

<xarray.DataArray (sample: 58587, xyt: 3)> Size: 1MB
array([[-1.98387097e+01,  1.75127334e+01,  0.00000000e+00],
       [-1.98387097e+01,  1.74448217e+01,  2.00000000e-02],
       [-1.98387097e+01,  1.73514431e+01,  4.00000000e-02],
       ...,
       [-1.18845501e+01,  8.40407470e+00,  1.20094000e+03],
       [-1.18845501e+01,  8.40407470e+00,  1.20096000e+03],
       [-1.18845501e+01,  8.40407470e+00,  1.20098000e+03]])
Coordinates:
  * sample   (sample) float32 234kB 0.0 1.0 2.0 ... 5.858e+04 5.859e+04
  * xyt      (xyt) float32 12B 0.0 1.0 2.0
Attributes:
    sample_rate:           50.0
    session_data_ref:      {'schema_ref': 'session', 'data_name': '1-13_20210...
    animal_data_ref:       {'schema_ref': 'animal', 'data_name': '1-13'}
    has_file:              true
    schema_ref:            animal_position
    data_name:             1-13_20210510_163938
    recording_length:      20210510
    data_dimensions:       ['sample', 'xyt']
    dimension_of_measure:  [space]
    unit_of_measure:       cm

In [16]:
type(session_xarrays[0]['pos_array'].attrs.get("sample_rate"))

str

In [3]:
filepath = '/Users/Anurag/Downloads/NeuroLab/Data/Output_Axona_Feb_12/input/1-13_20210510_110723_pos_array.nc'
pos_dataarray = xr.open_dataarray(filepath)
# pos_dataarray = xr.open_dataset(filepath)
pos_dataarray

<xarray.DataArray (sample: 57988, xyt: 3)> Size: 1MB
[173964 values with dtype=float64]
Coordinates:
  * sample   (sample) float32 232kB 0.0 1.0 2.0 ... 5.799e+04 5.799e+04
  * xyt      (xyt) float32 12B 0.0 1.0 2.0
Attributes:
    sample_rate:           50.0
    session_data_ref:      {'schema_ref': 'session', 'data_name': '1-13_20210...
    animal_data_ref:       {'schema_ref': 'animal', 'data_name': '1-13'}
    has_file:              true
    schema_ref:            animal_position
    data_name:             1-13_20210510_110723
    recording_length:      20210510
    data_dimensions:       ['sample', 'xyt']
    dimension_of_measure:  [space]
    unit_of_measure:       cm

In [ ]:
filepath = '/Users/Anurag/Downloads/NeuroLab/Data/Output_Axona_Feb_12/1-13_1_20210510_163938_tet_waveform_arrays.nc'
tet_dataarray = xr.open_dataarray(filepath)
tet_dataarray

<xarray.DataArray (spike_idx: 104237, channel: 4, sample: 50)> Size: 83MB
[20847400 values with dtype=float32]
Coordinates:
  * spike_idx  (spike_idx) float32 417kB 0.0 1.0 2.0 ... 1.042e+05 1.042e+05
  * channel    (channel) float32 16B 0.0 1.0 2.0 3.0
  * sample     (sample) float32 200B 0.0 1.0 2.0 3.0 4.0 ... 46.0 47.0 48.0 49.0
Attributes:
    session_data_ref:      {'schema_ref': 'session', 'data_name': '1-13_1_202...
    animal_data_ref:       {'schema_ref': 'animal', 'data_name': '1-13'}
    probe_data_ref:        {'schema_ref': 'probe', 'data_name': '1'}
    has_file:              true
    schema_ref:            tet_waveforms
    data_name:             1-13_1_20210510_163938
    data_dimensions:       ['spike_idx', 'channel', 'sample']
    dimension_of_measure:  [charge]

In [22]:
tet_dataarray.data.shape

(104237, 4, 50)

In [17]:
filepath = '/Users/Anurag/Downloads/NeuroLab/Data/Output_Axona_Feb_12/1-13_1_20210510_163938_cut_arrays.nc'
cut_dataarray = xr.open_dataarray(filepath)
# pos_dataarray = xr.open_dataset(filepath)
cut_dataarray

<xarray.DataArray (spike_idx: 104237, 1: 1)> Size: 417kB
[104237 values with dtype=float32]
Coordinates:
  * spike_idx  (spike_idx) int64 834kB 0 1 2 3 4 ... 104233 104234 104235 104236
Dimensions without coordinates: 1
Attributes:
    session_data_ref:      {'schema_ref': 'session', 'data_name': '1-13_1_202...
    animal_data_ref:       {'schema_ref': 'animal', 'data_name': '1-13'}
    probe_data_ref:        {'schema_ref': 'probe', 'data_name': '1'}
    has_file:              true
    schema_ref:            spike_labels
    data_name:             1-13_1_20210510_163938
    data_dimensions:       ['sample', 'xyt']
    dimension_of_measure:  [nominal]

In [21]:
cut_dataarray.shape

(104237, 1)

In [5]:
cut_dataarray.shape

(104237, 1)

In [6]:
import sys
sys.getsizeof(pos_dataarray)

104

In [7]:
pos_dataarray.nbytes

1391712

In [9]:
save_dir = "/Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf"
for session_xarrays in session_xarray_list:
    save_xarrays(session_xarrays, save_dir)

Saved pos_array to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_20210510_163938_pos_array.nc
Saved cut_arrays_1 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_1_20210510_163938_cut_arrays.nc
Saved cut_arrays_2 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_2_20210510_163938_cut_arrays.nc
Saved cut_arrays_3 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_3_20210510_163938_cut_arrays.nc
Saved cut_arrays_4 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_4_20210510_163938_cut_arrays.nc
Saved tet_waveform_arrays_1 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_1_20210510_163938_tet_waveform_arrays.nc
Saved tet_waveform_arrays_2 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_2_20210510_163938_tet_waveform_arrays.nc
Saved tet_waveform_arrays_3 to /Users/Anurag/Downloads/NeuroLab/Data/GusRemapData_Netcdf/1-13_3_20210510_163938_tet_waveform_arrays.nc
Saved tet_wavefo

In [ ]:
session_xarrays[0]['tet_waveform_arrays'][0]['sess']

<xarray.DataArray (spike_idx: 104237, channel: 4, sample: 50)> Size: 83MB
array([[[ 45.,  45.,  45., ...,   4.,   8.,  14.],
        [  0.,   0.,   7., ..., -14., -20., -23.],
        [ -8.,  -6., -12., ..., -21., -24., -23.],
        [ -7.,  -3.,  -1., ..., -14., -20., -22.]],

       [[ 45.,  45.,  45., ...,   4.,   8.,  14.],
        [  0.,   0.,   7., ..., -14., -20., -23.],
        [ -8.,  -6., -12., ..., -21., -24., -23.],
        [ -7.,  -3.,  -1., ..., -14., -20., -22.]],

       [[ 38.,  38.,  34., ...,  -5.,  -4.,   0.],
        [  4.,  19.,  19., ...,  25.,  25.,  31.],
        [ 18.,  21.,  19., ...,  -8.,  -8.,   5.],
        [ 17.,  20.,  19., ...,  30.,  30.,  32.]],

       ...,

       [[-38., -30., -19., ...,  -6., -16., -21.],
        [-18., -14.,  -9., ...,  -6., -13., -16.],
        [  1.,   0.,   0., ...,  -5.,  -9., -10.],
        [ -9.,  -4.,   0., ..., -18., -26., -28.]],

       [[-12.,   1.,  11., ..., -18., -14., -12.],
        [ -4.,   2.,  12., ..., -22., -25., -30.],
        [ 19.,  24.,  23., ..., -23., -27., -31.],
        [  5.,  11.,  13., ..., -38., -34., -28.]],

       [[ 22.,  25.,  28., ..., -72., -70., -76.],
        [ -5.,  -6., -16., ...,  -5., -14., -19.],
        [ -6., -11., -19., ...,   0.,   0.,   0.],
        [-14., -18., -18., ..., -12.,  -7.,  -5.]]], dtype=float32)
Coordinates:
  * spike_idx  (spike_idx) float32 417kB 0.0 1.0 2.0 ... 1.042e+05 1.042e+05
  * channel    (channel) float32 16B 0.0 1.0 2.0 3.0
  * sample     (sample) float32 200B 0.0 1.0 2.0 3.0 4.0 ... 46.0 47.0 48.0 49.0
Attributes:
    session_data_ref:      {'schema_ref': 'session', 'data_name': '1-13_20210...
    animal_data_ref:       {'schema_ref': 'animal', 'data_name': '1-13'}
    probe_data_ref:        {'schema_ref': 'probe', 'data_name': '1'}
    has_file:              true
    schema_ref:            tet_waveforms
    data_name:             1-13_1_20210510_163938
    data_dimensions:       ['spike_idx', 'channel', 'sample']
    dimension_of_measure:  [charge]